In [ ]:
import os
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

HOME_DIR = "/path/to/HOME_DIR"

TF_MAPPING_PATH = os.path.join(
    HOME_DIR,
    "data/Binary_dataV1/tf_mapping.csv"
)

UNIPROT_SOURCE_PATH = os.path.join(
    HOME_DIR,
    "data/tf_celltype_list_uniprotID_161_standardized.csv"
)

FEATURES_DIR = os.path.join(
    HOME_DIR,
    "Protein_data/prostt5_featuresV1"
)

OUTPUT_PATH = os.path.join(
    HOME_DIR,
    "data/Binary_dataV1/"
    "Binary_data_for_TF_splitV3/EGNN/"
    "tf_mapping_with_features_with_graphsV1.csv"
)


# ============================================================
# LOAD AUTHORITATIVE TF INDEX ORDER
# ============================================================

tf_mapping = pd.read_csv(TF_MAPPING_PATH)   # columns: tf_idx, TF

print(f"Loaded TF mapping: {tf_mapping.shape}")


# ============================================================
# LOAD TF → UNIPROT MAPPING
# ============================================================

uniprot_source = pd.read_csv(UNIPROT_SOURCE_PATH)

tf_to_uniprot = (
    uniprot_source[
        ["Transcription Factor", "UniProt ID"]
    ]
    .drop_duplicates(subset="Transcription Factor")
    .set_index("Transcription Factor")["UniProt ID"]
    .to_dict()
)


df = tf_mapping.copy()

df["uniprot_id"] = df["TF"].map(tf_to_uniprot)

resolved = df["uniprot_id"].notna().sum()

print(
    f"TFs with resolved UniProt ID: "
    f"{resolved}/{len(df)}"
)

missing = df[df["uniprot_id"].isna()]

if len(missing):
    print("\nMissing UniProt IDs:")
    print(missing[["tf_idx", "TF"]].to_string(index=False))


# ============================================================
# FIND CORRESPONDING PROSTT5 FEATURE FILE
# ============================================================

def find_matching_feature_file(
    tf_name,
    uniprot_id,
    features_dir
):

    special_mappings = {
        "Sin3Ak-20": "SIN3A",
        "Sin3A-20": "SIN3A",
        "Pol2-4H8": "Pol2",
        "PAX5-N19": "PAX5-C20",
        "GATA2-sc267": "GATA-2",
        "GATA2": "GATA-2",
        "eGFP-GATA2": "GATA-2",
        "eGFP-FOS": "c-Fos",
        "eGFP-JunD": "JunD",
        "TCF7L2_C9B9": "TCF7L2",
        "USF1": "USF-1",
        "UBTF": "UBF",
    }

    main_name = special_mappings.get(
        tf_name,
        tf_name
    )

    variations = [
        main_name,
        tf_name,
        tf_name.replace(" ", "-"),
        tf_name.replace(" ", "_"),
        tf_name.replace("-", "_"),
    ]

    # Remove duplicate names while keeping order
    variations = list(dict.fromkeys(variations))

    # Also try uppercase versions
    variations += [
        name.upper()
        for name in variations
    ]

    variations = list(dict.fromkeys(variations))

    for variation in variations:

        candidates = [
            f"{variation}_{uniprot_id}.fea",
            f"{variation}_{str(uniprot_id).lower()}.fea",
        ]

        for candidate in candidates:

            path = os.path.join(
                features_dir,
                candidate
            )

            if os.path.exists(path):
                return candidate, "base_name_match"

    return None, "no_match"


# ============================================================
# MAP ALL TFs TO THEIR PROSTT5 FEATURES
# ============================================================

matched_filenames = []
match_methods = []

for _, row in df.iterrows():

    if pd.isna(row["uniprot_id"]):

        matched_filenames.append(None)
        match_methods.append("no_uniprot_id")

        continue

    filename, method = find_matching_feature_file(
        row["TF"],
        row["uniprot_id"],
        FEATURES_DIR
    )

    matched_filenames.append(filename)
    match_methods.append(method)


df["matched_filename"] = matched_filenames
df["match_method"] = match_methods

# Training loader can combine this filename with FEATURES_DIR
df["prot5_feature_path"] = df["matched_filename"]

# Not used in the current ProstT5 model
df["graph_filename"] = None


# ============================================================
# REPORT RESULTS
# ============================================================

matched = df["matched_filename"].notna().sum()

print(
    f"\nMatched ProstT5 features: "
    f"{matched}/{len(df)}"
)

unmatched = df[
    df["matched_filename"].isna()
]

if len(unmatched):

    print("\nUnmatched TFs:")

    print(
        unmatched[
            ["tf_idx", "TF", "uniprot_id"]
        ].to_string(index=False)
    )


# ============================================================
# VALIDATION
# ============================================================

assert len(df) == 161, \
    f"Expected 161 TFs, found {len(df)}"

assert df["tf_idx"].tolist() == list(range(161)), \
    "tf_idx ordering is not 0-160"

assert df["uniprot_id"].notna().all(), \
    "Some TFs do not have UniProt IDs"

assert df["matched_filename"].notna().all(), \
    "Some TFs do not have matching ProstT5 feature files"

print("\nPASS: All 161 TFs successfully mapped.")
print("PASS: tf_idx order preserved from 0 to 160.")


# ============================================================
# SAVE FINAL MAPPING
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_PATH),
    exist_ok=True
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"\nSaved mapping to:")
print(OUTPUT_PATH)


# ============================================================
# SHOW FIRST FEW ROWS
# ============================================================

print("\nFirst 10 mappings:")

print(
    df[
        [
            "tf_idx",
            "TF",
            "uniprot_id",
            "matched_filename",
            "match_method"
        ]
    ]
    .head(10)
    .to_string(index=False)
)